In [ ]:
!curl -O https://raw.githubusercontent.com/gentleman644/foxtrot-thyroid-cancer-prediction/refs/heads/main/UnitTest.py
# unitTestPath = os.path.join(os.getcwd(), "UnitTest.py")
%run -i "UnitTest.py"

# Imports and load dataset
import importlib
import UnitTest
importlib.reload(UnitTest)

from UnitTest import TestMethods as tm
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Load dataset
dataset = pd.read_csv("https://github.com/gentleman644/foxtrot-thyroid-cancer-prediction/blob/main/thyroid_cancer_risk_data.csv?raw=true",
            header=0, index_col=0)

# Preview dataset 1
print(dataset.head(), "\n") #first 5 rows
print(dataset.tail(), "\n") #last 5 rows
tm.test_load_data(dataset)

In [ ]:
from pandas.api.types import is_numeric_dtype
# Preview data features and figure out problems for data preprocessing
# P1: 11 non-numeric features, which need to be encoded before feeding into ML models
# P2: Class imbalance in the target variable (Diagnosis), which requires oversampling
print(dataset.info(), "\n")
print(dataset.describe(), "\n")

print("Count of each non-numeric feature:")
for col in dataset.columns:
    if (is_numeric_dtype(dataset[col]) == False):
        print(f"{col:<24}({dataset[col].dtype}): {len(dataset[col])}")

print("\nCount of each class in the target variable:")
print(dataset["Diagnosis"].value_counts())

## Data Preprocessing - Feature Encoding
*Write any notes or observations about the feature encoding process here if you want*

In [ ]:
# P1: Encode all non-numeric features (use OrdinalEncoder)
# (Transform all non-numeric features into numeric features, so that we can feed them into ML models)
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder(categories="auto")
dataset_encoded = dataset.copy()
for col in dataset.columns:
    if (is_numeric_dtype(dataset[col]) == False):
        dataset_encoded[col] = ordinal_encoder.fit_transform(dataset[[col]])

print(dataset_encoded.info(), "\n")
print(dataset_encoded.head(), "\n")
print(dataset_encoded.tail(), "\n")

print(f"{'Category':<13} | {'Count':<8} | {'Encoded':<6} | {'Count':<8}")
for col in dataset.columns:
    if (is_numeric_dtype(dataset[col]) == False):
        # Print value counts of original vs encoded, one line each after the other
        print(f"--{col}--")
        original_counts = dataset[col].value_counts().to_dict()
        encoded_counts = dataset_encoded[col].value_counts().to_dict()
        for category in original_counts.keys():
            original_count = original_counts[category]
            encoded_category = dataset_encoded[col].loc[dataset[col].index[dataset[col] == category][0]]
            encoded_count = encoded_counts[encoded_category] if encoded_category is not None else None
            print(f"{str(category):<14}: {original_count:<8} | {str(encoded_category):<8}: {encoded_count:<6}")

dataset = dataset_encoded.copy()
tm.test_ordinal_encoder(dataset)


## Data Preprocessing - Target Variable Class Imbalance
*Write any notes or observations about the target variable class imbalance process here if you want*

In [ ]:
# P2: Class imbalance in the target variable (Diagnosis).
# Keep all rows (do NOT undersample) to preserve signal for learning.

# 1. Verify columns & class imbalance
print(f"Starting column is '{dataset.columns[0]}'")
print(f"Target variable is '{dataset.columns[15]}'")
print(dataset['Diagnosis'].value_counts())

# 2. Separate features and target variable
feats_X = dataset.iloc[:, :15].copy()
trgt_y = dataset.iloc[:, 15].copy()

# 3. Recombine into a single dataframe without resampling
dataset = pd.concat([feats_X, trgt_y], axis=1)

# 4. Verify results
print("Class distribution kept (no undersampling):")
print(trgt_y.value_counts())
print(dataset.info())
print(dataset.head())

# didn't need to resample because we kept all rows, so don't test this
# tm.test_target_class_imbalance(resmpl_y)

## Data visualization - Preview dataset
*Write any notes or observations about the dataset preview here if you want*

Determine any skewness in the dataset, and any problems that need to be handled in data preprocessing.

In [ ]:
# Part 4: Data graphs

# Graphical: Preview dataset 1
import matplotlib.pyplot as plt

dataset.hist(bins=75, figsize=(14, 13))
plt.show()

# Graphical: Preview dataset 2
from pandas.plotting import scatter_matrix
attributes = []
for col in dataset.columns:
    attributes.append(col)
# scatter matrix takes a long time to run so commented out for now
# scatter_matrix(dataset[attributes], alpha=0.9, figsize=(17, 15), diagonal='kde')

In [ ]:
# Feature engineering steps

# Feature Identification ------
# Convert dataframe to float type
# Select input and target features (feats_X and trgt_y)

dataset = dataset.astype("float64")  # Convert pandas-dataframe to "float" type
feats_X = dataset.iloc[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]]
print("Features (Inputs or Independent variables):", feats_X.shape, "\n")

trgt_y = pd.DataFrame(dataset.iloc[:, 15])
print("Targets (Outputs or Dependent variables):", trgt_y.shape)
tm.test_feature_Identification(feats_X, trgt_y)

In [ ]:
# Feature Identification ------
# Convert dataframe to float type
# Select input and target features (feats_X and trgt_y)

dataset = dataset.astype("float64")  # Convert pandas-dataframe to "float" type
feats_X = dataset.iloc[:, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]]
trgt_y = pd.DataFrame(dataset.iloc[:, 15])

# Feature Generation ------
# Create new features based on existing features (add new features to feats_X[])
# Check for 'NaN' or 'inf' values in features/inputs (Independent variables)
base_feats = ["TSH_Level", "T3_Level", "T4_Level", "Nodule_Size"]
new_feats = []
temp_df = feats_X.copy()
for col in dataset.columns:
    for i in range(len(base_feats)):
        if (col in base_feats and col != "Diagnosis" and col != base_feats[i]):
            # don't add inf or nan values
            # temp_df = abs(feats_X[col] / feats_X[base_feats[i]])
            # if not np.isinf(temp_df[f"{col}_per_{base_feats[i]}"][0]) and not np.isnan(temp_df[f"{col}_per_{base_feats[i]}"][0]):
            new_feats.append(f"{col}_per_{base_feats[i]}")
            feats_X[f"{col}_per_{base_feats[i]}"] = abs(feats_X[col] / feats_X[base_feats[i]])

print(new_feats, "\n")
print(base_feats, "\n")

for col in dataset.columns:
    for i in range(len(base_feats)):
        if (col in base_feats and col != "Diagnosis" and col != base_feats[i]):
            # don't add inf or nan values
            # temp_df = abs(feats_X[col] / feats_X[base_feats[i]])
            # if not np.isinf(temp_df[f"{col}_per_{base_feats[i]}"][0]) and not np.isnan(temp_df[f"{col}_per_{base_feats[i]}"][0]):
            new_feats.append(f"{col}_by_{base_feats[i]}")
            feats_X[f"{col}_by_{base_feats[i]}"] = feats_X[col] * feats_X[base_feats[i]]

indices = np.where(np.isinf(feats_X) | np.isnan(feats_X))
print("'NaN' or 'inf' indices (row, column): ", list(zip(*indices)))

# remove inf and nan values from dataframe
feats_X = feats_X.dropna()
feats_X = feats_X.dropna(axis=1)
for col in feats_X.columns:
    if feats_X[col].isnull().values.any() or np.isinf(feats_X[col]).any():
        feats_X = feats_X.drop(col, axis=1)

indices = np.where(np.isinf(feats_X) | np.isnan(feats_X))
print("'NaN' or 'inf' indices (row, column): ", list(zip(*indices)))

feats_X.hist(bins=75, figsize=(14, 13))
plt.show()

print(feats_X.head(), "\n")
print(feats_X.tail(), "\n")
print(feats_X.info(), "\n")


# This should now safely print an empty list: []
indices = np.where(np.isinf(feats_X) | np.isnan(feats_X))
print("'NaN' or 'inf' indices (row, column): ", list(zip(*indices)))
tm.test_feature_generation(feats_X)

In [ ]:
# Feature Engineering -> Feature Selection
# Relevant/Good: features(Independent/Input) --high +/-ve correlation--> target(Dependent/Output)
# Irrelevant/Bad: features(Independent/Input) --low near-0 correlation--> target(Dependent/Output)
import seaborn as sbn
from sklearn.feature_selection import SelectKBest, mutual_info_classif

#Print the scores of all features in relation to target variable

for t in range(trgt_y.shape[1]):
    # 1. Fit the KBest selector
    KBest = SelectKBest(score_func=mutual_info_classif, k=30)   # Linear(f_regression|f_classif) to target; Nonlinear(mutual_info_regression|mutual_info_classif) to target
    output = KBest.fit_transform(feats_X, trgt_y.iloc[:,t])  # Fits to data & transform/reduce it to the selected features
    top_feats_X = pd.DataFrame(output, columns=KBest.get_feature_names_out())   # Transfrom numpy-array 2 pandas-dataframe
    
   # 2. Create Scroreboard to help Selection due to large number of features
    target_name = trgt_y.columns[t] 
    scores_df = pd.DataFrame({
        'Feature': feats_X.columns,
        'Score': KBest.scores_
    })
    
    # Sort the scoreboard from highest (best) to lowest (worst)
    scores_df = scores_df.sort_values(by='Score', ascending=False)
print(scores_df.head(12))


# AVOID Multi-Collinearity/Redundancy: features(Independent/Input) --high +/-ve correlation--> features(Independent/Input)
useful_feats_X = top_feats_X.loc[:, :]
useful_feats_corr = useful_feats_X.corr('kendall')  # Linear(pearson) to target; Nonlinear(kendall|spearman) to target
plt.figure(figsize=(18, 14))
sbn.heatmap(useful_feats_corr, annot=True, linewidths=0.5, fmt='.2f', cmap='coolwarm')  # cmap='coolwarm|RdBu|BrBG|PRGn|viridis|Blues|Greens|Oranges|rocket|tab10|Set1|Pastel1|Accent'
plt.show()  # figsize(inch, inch); (-ve corr., 0, +ve corr.)

print()
#Reshape the heatmap to focus on the top 6 features with highest scores and lowest correlation with each other
# Instead of doing this manually, use a loop and check correlation scores and drop those above a certain threshold
temp_df = top_feats_X.copy()
feats_to_not_use = []

# drop features with high correlation to other features (multi-collinearity/redundancy)
for col in useful_feats_corr.columns:
    for idx in useful_feats_corr.index:
        if (col != idx and abs(useful_feats_corr.loc[idx, col]) > 0.6):
            if (col not in feats_to_not_use and idx not in feats_to_not_use):
                print(f"Correlation between '{col}' and '{idx}': {useful_feats_corr.loc[idx, col]:.2f}")
                feats_to_not_use.append(col)
                break


temp_df = temp_df.drop(columns=feats_to_not_use)
useful_feats_X = temp_df.copy()
useful_feats_corr = useful_feats_X.corr('kendall')  # Linear(pearson) to target; Nonlinear(kendall|spearman) to target
plt.figure(figsize=(16, 12))
sbn.heatmap(useful_feats_corr, annot=True, linewidths=0.5, fmt='.2f', cmap='coolwarm')  # cmap='coolwarm|RdBu|BrBG|PRGn|viridis|Blues|Greens|Oranges|rocket|tab10|Set1|Pastel1|Accent'
plt.show()  # figsize(inch, inch); (-ve corr., 0, +ve corr.)
tm.test_feature_Selection(useful_feats_X)

In [ ]:
# Training & Test Sets: StratifiedKFold + ShuffleSplit
from sklearn.model_selection import StratifiedShuffleSplit, ShuffleSplit

X = useful_feats_X
y = trgt_y
print(X.shape)
print(y.shape, "\n")

strat_data = StratifiedShuffleSplit(n_splits=1, test_size=0.2, train_size=0.8)
for train_index, test_index in strat_data.split(X, y):
  X_train, y_train = X.iloc[train_index], y.iloc[train_index]
  X_test, y_test = X.iloc[test_index], y.iloc[test_index]
print("X_train:", X_train.shape, ",\t", "y_train:", y_train.shape)
print("X_test:", X_test.shape, ",\t", "y_test:", y_test.shape)

In [ ]:
# Fix P3: Feature Transformation/Scaling/Normalization of Features or Samples ("skewness/unsymmetric" dataset)
from sklearn.preprocessing import QuantileTransformer, StandardScaler, MinMaxScaler, Normalizer
import pandas as pd
import matplotlib.pyplot as plt

# Columns to split
nominal_cols = ["Gender", "Country", "Ethnicity", "Family_History",
                "Radiation_Exposure", "Iodine_Deficiency", "Diabetes",
                "Thyroid_Cancer_Risk"]

X_train = pd.get_dummies(X_train, columns=nominal_cols, drop_first=True, dtype=float)
X_test = pd.get_dummies(X_test, columns=nominal_cols, drop_first=True, dtype=float)

# CRITICAL ALIGNMENT
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0.0)

In [ ]:
# Z-Score Normalization (or Standardization) wrt. Features/Columns: Non-Linear Transformation for 'feats_X'
quantile_trans_standzatn = QuantileTransformer(output_distribution='uniform')
quantile_trans_standzatn = quantile_trans_standzatn.fit(X_train)  # Fit ONLY on Train

standized_X_train = quantile_trans_standzatn.transform(X_train)
standized_X_test = quantile_trans_standzatn.transform(X_test)

# Z-Score Normalization (or Standardization) wrt. Features/Columns: Linear Transformation for 'trgt_y'
standized_y_train = y_train
standized_y_test = y_test

# Histrogram shows dataset WITHOUT 'skewness' (data is uniformly distributed)
pd.DataFrame(standized_X_train, columns=X_train.columns).hist(bins=50, figsize=(15, 14))
plt.tight_layout() 
plt.show()
tm.test_feature_normalization(standized_X_train, standized_y_train, standized_X_test, standized_y_test)

In [ ]:
# Algorithm Selection ----
# Import possible ML models
# Select ML model to test/use
# Select ML/DL algorithm AND Tune/Re-tune hyperparameters
from sklearn.multioutput import MultiOutputRegressor, MultiOutputClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

ml_algorithm = "Bernoulli_N_Bayes"  # KNN, GradBoost, RandFrst, ExtraTree, DeciTree, etc

# Create if/then path to automatically import model and apply hyperparameters
if (ml_algorithm == "LinearRegress"):
  algorithm = LinearRegression()  # For regression only
elif (ml_algorithm == "LogisticRegress"):
  algorithm = LogisticRegression(solver='lbfgs')  # For classification only
elif (ml_algorithm == "SuppVec"):
  algorithm = SVC()
elif (ml_algorithm == "StochastGrad"):
  algorithm = SGDClassifier(learning_rate='invscaling', eta0=0.01)
elif (ml_algorithm == "Gaussian_N_Bayes"):
  algorithm = GaussianNB()
elif (ml_algorithm == "Multinomial_N_Bayes"):
  algorithm = MultinomialNB()
elif (ml_algorithm == "Bernoulli_N_Bayes"):
  algorithm = BernoulliNB()
elif (ml_algorithm == "Categorical_N_Bayes"):
  algorithm = CategoricalNB()
elif (ml_algorithm == "GradBoost"):
  algorithm = GradientBoostingClassifier()
elif (ml_algorithm == "DeciTree"):
  algorithm = DecisionTreeClassifier()
elif (ml_algorithm == "RandFrst"):
  algorithm = RandomForestClassifier(n_estimators=100)
elif (ml_algorithm == "ExtraTree"):
  algorithm = ExtraTreesClassifier(n_estimators=100)
elif (ml_algorithm == "KNN"):
  algorithm = KNeighborsClassifier(n_neighbors=6, weights='distance')
elif (ml_algorithm == "DL_MLP"):
  algorithm = MLPClassifier(hidden_layer_sizes=(standized_X_train.shape[1], 30, 50, 10, standized_y_train.shape[1]), activation='relu', solver='lbfgs', learning_rate='constant', learning_rate_init=0.001)

In [ ]:

# TRAINING: K-Fold Cross-Validation. 
#  algorithm(ML) to data(Training).
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, matthews_corrcoef

# Get probabilities instead of direct predictions
cv_preds_proba = cross_val_predict(algorithm, standized_X_train, standized_y_train.values.ravel(), cv=3, method='predict_proba')
print("Shape of Output/Probabilities: ", cv_preds_proba.shape, "\n")

#Tune threshold to prioritize recall# Get probabilities instead of direct predictions
cv_preds_proba = cross_val_predict(
    algorithm,
    standized_X_train,
    standized_y_train.values.ravel(),
    cv=3,
    method='predict_proba'
)
print("Shape of Output/Probabilities: ", cv_preds_proba.shape, "\n")

# Tune threshold on CV probabilities (maximize F1 or accuracy)
raw_y_train = y_train.values.ravel()

thresholds = np.arange(0.30, 0.71, 0.01)
best_threshold = 0.5
best_f1 = -1

for t in thresholds:
    preds_t = (cv_preds_proba[:, 1] > t).astype(int)
    f1_t = precision_recall_fscore_support(raw_y_train, preds_t, average='weighted')[2]
    # f1_t = accuracy_score(raw_y_train, preds_t)
    if f1_t > best_f1:
        best_f1 = f1_t
        best_threshold = t

print(f"Best threshold: {best_threshold:.2f}, Best CV weighted F1: {best_f1:.4f}")

# Use best threshold
cv_preds = (cv_preds_proba[:, 1] > best_threshold).astype(int)
reversed_y_train = np.rint(cv_preds).astype(np.int32)

# Evaluate model's performance. Overfitting(Train ERR << Test ERR); Underfitting(Train ERR >> Test ERR).
raw_y_train = y_train.values.ravel()
acc = accuracy_score(raw_y_train, reversed_y_train, normalize=True)  # normalize=True(fraction of correctly classified samples); normalize=False(no. of correctly classified samples)
pr_rc_fs_sp = precision_recall_fscore_support(raw_y_train, reversed_y_train, average='weighted')  # average='weighted'(compute metrics per label, and find their avg. weighted by support (no. of true instances per label))
mcc = matthews_corrcoef(raw_y_train, reversed_y_train)
# Note: ROC AUC uses probabilities directly, no threshold needed
roc = roc_auc_score(raw_y_train, cv_preds_proba[:, 1], average='weighted', multi_class='ovo')  # multi_class='ovo'(meaning One-vs-one. Computes avg. AUC of all possible pairwise combinations of classes)

# Training/Validation Metrics
print("Training Metrics: ", "\n----------------")
print("ACCURACY: ", acc)
print("PRECISION: ", pr_rc_fs_sp[0])
print("RECALL: ", pr_rc_fs_sp[1])  # This should be higher now
print("F1-SCORE: ", pr_rc_fs_sp[2])
print("AREA under ROC: ", roc)
print("MCC: ", mcc)

In [ ]:
# TESTING/GENERALIZATION: Make Predictions for 'y-component' wrt. data(Test)
model = algorithm
model.fit(standized_X_train, standized_y_train.values.ravel())

# Get probabilities for test set
pred_proba_test = model.predict_proba(standized_X_test)

threshold = best_threshold  # Match the training threshold
pred_y_test = (pred_proba_test[:, 1] > threshold).astype(int)

reversed_y_test = pd.DataFrame(pred_y_test)
reversed_y_test = np.rint(reversed_y_test).astype(np.int32)

# Evaluate model's performance. Overfitting(Train ERR << Test ERR); Underfitting(Train ERR >> Test ERR).
raw_y_test = y_test.values.ravel()
acc = accuracy_score(raw_y_test, reversed_y_test, normalize=True)  # normalize=True(fraction of correctly classified samples); normalize=False(no. of correctly classified samples)
pr_rc_fs_sp = precision_recall_fscore_support(raw_y_test, reversed_y_test, average='weighted')  # average='weighted'(compute metrics per label, and find their avg. weighted by support (no. of true instances per label))
mcc = matthews_corrcoef(raw_y_test, reversed_y_test)
# ROC AUC uses probabilities
roc = roc_auc_score(raw_y_test, pred_proba_test[:, 1], average='weighted', multi_class='ovo')  # multi_class='ovo'(meaning One-vs-one. Computes avg. AUC of all possible pairwise combinations of classes)

# Testing Metrics
print("Testing Metrics: ", "\n----------------")
print("ACCURACY: ", acc)
print("PRECISION: ", pr_rc_fs_sp[0])
print("RECALL: ", pr_rc_fs_sp[1])  # This should be higher now
print("F1-SCORE: ", pr_rc_fs_sp[2])
print("AREA under ROC: ", roc)
print("MCC: ", mcc)

# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(raw_y_test, reversed_y_test)
classes = [0, 1]
ax = sbn.heatmap(cm, annot=True, fmt="d", cmap="Grays", cbar=True, linewidths=0.7, linecolor="black", square=True)
ax.set_xlabel("Predicted (Polarity) Labels")
ax.set_ylabel("Groundtruth (Polarity) Labels")

In [ ]:
#SAVE MODEL 
from joblib import dump, load

dump(model, "model.joblib");
dump(quantile_trans_standzatn, "standardization.joblib");

In [2]:
#DEMO
from joblib import load

import pandas as pd
import numpy as np
from sklearn.preprocessing import QuantileTransformer


current_model = load("model.joblib")
standard = load("standardization.joblib")

Age = 76
Smoking = True
Obesity = True
Family_History = True
Gender = 'Male'
Thyroid_Cancer_Risk = 'High'
T4_Level = 5.0
T3_Level = 1.0
TSH_Level = 3.0
Nodule_Size = 2.0
Country = "Russia"
Ethnicity = "Asian"
Radiation_Exposure = True
Iodine_Deficiency = True
Diabetes = True

predict_input = pd.DataFrame([{
    "Age" : int(Age),
    "Smoking" : int(Smoking),
    "Obesity" : int(Obesity),
    "TSH_Level_per_T3_Level": TSH_Level / T3_Level,
    "T4_Level_per_T3_Level": T4_Level / T3_Level,
    "Nodule_Size_per_TSH_Level" : Nodule_Size / TSH_Level,
    "Nodule_Size_per_T3_Level" : Nodule_Size / T3_Level,
    "Nodule_Size_per_T4_Level" : Nodule_Size / T4_Level,
    "T3_Level_by_TSH_Level" : T3_Level / TSH_Level,
    "T4_Level_by_TSH_Level" : T4_Level / TSH_Level,
    "T4_Level_by_T3_Level" : T4_Level / T3_Level,
    "Nodule_Size_by_TSH_Level" : Nodule_Size / TSH_Level,
    "Nodule_Size_by_T3_Level" : Nodule_Size / T3_Level,
    "Nodule_Size_by_T4_Level" : Nodule_Size / T4_Level,
    "Gender_1.0" : 0,
    "Country_1.0" : 0,
    "Country_2.0" : 0,
    "Country_3.0" : 0,
    "Country_4.0" : 0,
    "Country_5.0" : 0,
    "Country_6.0" : 0,
    "Country_7.0" : 0,
    "Country_8.0" : 0,
    "Country_9.0" : 0,
    "Ethnicity_1.0" : 0,
    "Ethnicity_2.0" : 0,
    "Ethnicity_3.0" : 0,
    "Ethnicity_4.0" : 0,
    "Family_History_1.0" : int(Family_History),
    "Radiation_Exposure_1.0" : int(Radiation_Exposure),
    "Iodine_Deficiency_1.0" : int(Iodine_Deficiency),
    "Diabetes_1.0" : int(Diabetes),
    "Thyroid_Cancer_Risk_1.0" : 0,
    "Thyroid_Cancer_Risk_2.0" : 0
}])

match Gender:
    case 'Male'| 'male':
        predict_input['Gender_1.0'] = True
    case 'Female' | 'female':
        predict_input['Gender_1.0'] = False

match Country:
    case 'Germany':
        predict_input['Country_1.0'] = 1
    case 'Nigeria':
        predict_input["Country_2.0"] = 1
    case 'India':
        predict_input["Country_3.0"] = 1
    case 'UK':
        predict_input["Country_4.0"] = 1
    case 'South Korea':
        predict_input["Country_5.0"] = 1
    case 'Brazil':
        predict_input["Country_6.0"] = 1
    case 'China':
        predict_input["Country_7.0"] = 1
    case 'US':
        predict_input["Country_8.0"] = 1
    case 'Japan':
        predict_input["Country_9.0"] = 1

match Ethnicity:
    case 'Hispanic':
        predict_input['Ethnicity_1.0'] = 1
    case 'Asian':
        predict_input['Ethnicity_2.0'] = 1
    case 'African':
        predict_input['Ethnicity_3.0'] = 1
    case 'Middle Eastern':
        predict_input['Ethnicity_4.0'] = 1

match Thyroid_Cancer_Risk:
    case 'Low':
        predict_input['Thyroid_Cancer_Risk_1.0'] = 1
    case 'Medium':
        predict_input['Thyroid_Cancer_Risk_2.0'] = 1

standized_predict_input = standard.transform(predict_input)
predict_output = current_model.predict_proba(standized_predict_input)
print("Malignant" if (predict_output[:,1] > 0.4) else "Benign")

Malignant
